# 06 — Bhaduri 2021 Download & Raw AnnData

Downloads the Bhaduri et al. 2021 *An Atlas of Cortical Arealization* fetal cortex dataset
(Nature 2021, PMID 34616070) from the NeMO archive, merges UCSC Cell Browser
annotations, and saves a single raw AnnData on Drive.

**This notebook replaces Zhong 2018 as the fetal reference.** Motivation: Zhong (2,394 cells,
Smart-seq2) caused DPT trajectory failure due to extreme 100× imbalance and disconnected
graph components (see `outputs_local/colab_05_trajectory_DIAGNOSTIC_WITH_OUTPUT.ipynb`).

**Rationale for Bhaduri 2021:**
- Same lab + same 10x Chromium v2 chemistry as Bhaduri 2020 organoids → batch effects minimised
- 11 donors GW14–25, multiple cortical areas → covers full organoid developmental window
- Open 10x MEX tarballs on NeMO (raw FASTQs restricted on dbGaP, processed counts openly hosted)
- UCSC Cell Browser exposes per-cell annotations → no need to re-cluster/re-annotate

**Coverage:** 74 samples / 83 NeMO folders, ~396,202 annotated cells (98% of UCSC's 404k
neocortex subset). One UCSC sample (`GW18_temporal`, 8,016 cells) has no NeMO folder and is dropped —
loss is absorbed because GW18 still has 4 other areas and GW18_2 covers temporal at the same GW.

**Run once.** After this, `colab_07_preprocessing.ipynb` loads the saved h5ad.


## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Paths and Config

In [2]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'
DATASET_NAME = 'bhaduri_2021'

RAW_DIR       = os.path.join(DRIVE_ROOT, 'data', 'raw',       DATASET_NAME)
PROCESSED_DIR = os.path.join(DRIVE_ROOT, 'data', 'processed', DATASET_NAME)
os.makedirs(RAW_DIR,       exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Colab ephemeral workspace — tarballs and extracted MEX files live here, not on Drive.
# 3.3 GB of tarballs doesn't need to be on Drive (it's regeneratable).
WORK_DIR    = '/content/bhaduri2021'
TARBALL_DIR = os.path.join(WORK_DIR, 'tarballs')
EXTRACT_DIR = os.path.join(WORK_DIR, 'extracted')
os.makedirs(TARBALL_DIR, exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

OUT_H5AD = os.path.join(PROCESSED_DIR, 'bhaduri_2021_raw.h5ad')
UCSC_META_URL = 'https://cells.ucsc.edu/dev-brain-regions/neocortex/meta.tsv'
UCSC_META_LOCAL = os.path.join(WORK_DIR, 'ucsc_neocortex_meta.tsv')

print(f'Drive output:  {OUT_H5AD}')
print(f'Work dir:      {WORK_DIR}')

Drive output:  /content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2021/bhaduri_2021_raw.h5ad
Work dir:      /content/bhaduri2021


## 3. Install Dependencies

In [3]:
!pip install -q scanpy

## 4. Sample Mapping

Inline table: UCSC sample prefix → list of NeMO folders (each with its resolved tarball URL).
All 83 URLs were verified by HEAD-probe during authoring (3 URL conventions on NeMO).

10 samples from GW22 / GW22T / GW22L are split into `_1` / `_2` lane pairs on NeMO
and get merged into a single UCSC sample during loading.


In [4]:
# Auto-generated: UCSC prefix -> list of (NeMO folder, tarball URL)
# Total: 74 samples
SAMPLES = {
    'GW14_V1': {  # 1988 cells
        'nemo': [
            ('GW14_occipital', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW14_occipital/GRCh38/GRCh38.mex.tar.gz'),
        ],
    },
    'GW14_motor': {  # 560 cells
        'nemo': [
            ('GW14_motor', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW14_motor/GRCh38/GRCh38.mex.tar.gz'),
        ],
    },
    'GW14_somatosensory': {  # 5480 cells
        'nemo': [
            ('GW14_somato', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW14_somato/GRCh38/GRCh38.mex.tar.gz'),
        ],
    },
    'GW18_2_ParietalVZ': {  # 9227 cells
        'nemo': [
            ('GW18_2_ParVZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_2_ParVZ/GW18_2_ParVZ.mex.tar.gz'),
        ],
    },
    'GW18_2_TemporalVZ': {  # 8747 cells
        'nemo': [
            ('GW18_2_TempVZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_2_TempVZ/GW18_2_TempVZ.mex.tar.gz'),
        ],
    },
    'GW18_2_V1VZ': {  # 10743 cells
        'nemo': [
            ('GW18_2_V1VZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_2_V1VZ/GW18_2_V1VZ.mex.tar.gz'),
        ],
    },
    'GW18_2_motor': {  # 9293 cells
        'nemo': [
            ('GW18_2_motor', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_2_motor/GW18_2_motor.mex.tar.gz'),
        ],
    },
    'GW18_2_motorVZ': {  # 4534 cells
        'nemo': [
            ('GW18_2_motorVZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_2_motorVZ/GW18_2_motorVZ.mex.tar.gz'),
        ],
    },
    'GW18_2_parietal': {  # 6905 cells
        'nemo': [
            ('GW18_2_parietal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_2_parietal/GW18_2_parietal.mex.tar.gz'),
        ],
    },
    'GW18_2_somato': {  # 6302 cells
        'nemo': [
            ('GW18_2_somato', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_2_somato/GW18_2_somato.mex.tar.gz'),
        ],
    },
    'GW18_2_somatoVZ': {  # 4212 cells
        'nemo': [
            ('GW18_2_somatoVZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_2_somatoVZ/GW18_2_somatoVZ.mex.tar.gz'),
        ],
    },
    'GW18_2_temporal': {  # 10103 cells
        'nemo': [
            ('GW18_2_temporal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_2_temporal/GW18_2_temporal.mex.tar.gz'),
        ],
    },
    'GW18_PFC': {  # 13786 cells
        'nemo': [
            ('GW18_PFC', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_PFC/GW18_PFC.mex.tar.gz'),
        ],
    },
    'GW18_V1': {  # 10079 cells
        'nemo': [
            ('GW18_V1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_V1/GW18_V1.mex.tar.gz'),
        ],
    },
    'GW18_motor': {  # 14619 cells
        'nemo': [
            ('GW18_motor', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_motor/GW18_motor.mex.tar.gz'),
        ],
    },
    'GW18_parietal': {  # 7929 cells
        'nemo': [
            ('GW18_parietal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW18_parietal/GW18_parietal.mex.tar.gz'),
        ],
    },
    'GW19_PFC_CP': {  # 3731 cells
        'nemo': [
            ('GW19_PFC_CP', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_PFC_CP/GW19_PFC_CP.mex.tar.gz'),
        ],
    },
    'GW19_PFC_all': {  # 3565 cells
        'nemo': [
            ('GW19_PFC_all', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_PFC_all/GW19_PFC_all.mex.tar.gz'),
        ],
    },
    'GW19_V1_CP': {  # 1153 cells
        'nemo': [
            ('GW19_V1_CP', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_V1_CP/GW19_V1_CP.mex.tar.gz'),
        ],
    },
    'GW19_V1_all': {  # 3939 cells
        'nemo': [
            ('GW19_V1_all', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_V1_all/GW19_V1_all.mex.tar.gz'),
        ],
    },
    'GW19_motor_CP': {  # 1593 cells
        'nemo': [
            ('GW19_M1_CP', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_M1_CP/GW19_M1_CP.mex.tar.gz'),
        ],
    },
    'GW19_motor_all': {  # 5075 cells
        'nemo': [
            ('GW19_M1_all', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_M1_all/GW19_M1_all.mex.tar.gz'),
        ],
    },
    'GW19_parietal': {  # 5078 cells
        'nemo': [
            ('GW19_Parietal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_Parietal/GW19_Parietal.mex.tar.gz'),
        ],
    },
    'GW19_somatosensory': {  # 5407 cells
        'nemo': [
            ('GW19_S1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_S1/GW19_S1.mex.tar.gz'),
        ],
    },
    'GW19_temporal': {  # 4328 cells
        'nemo': [
            ('GW19_Temp_all', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_Temp_all/GW19_Temp_all.mex.tar.gz'),
        ],
    },
    'GW2031_PFC': {  # 6681 cells
        'nemo': [
            ('GW20_31_PFC', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_31_PFC/GW20_31_PFC.mex.tar.gz'),
        ],
    },
    'GW2031_V1': {  # 3808 cells
        'nemo': [
            ('GW20_31_V1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_31_V1/GW20_31_V1.mex.tar.gz'),
        ],
    },
    'GW2031_parietal': {  # 3023 cells
        'nemo': [
            ('GW20_31_parietal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_31_parietal/GW20_31_parietal.mex.tar.gz'),
        ],
    },
    'GW2031_temporal': {  # 5177 cells
        'nemo': [
            ('GW20_31_temporal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_31_temporal/GW20_31_temporal.mex.tar.gz'),
        ],
    },
    'GW2034_PFC': {  # 3243 cells
        'nemo': [
            ('GW20_34_PFC', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_34_PFC/GW20_34_PFC.mex.tar.gz'),
        ],
    },
    'GW2034_PFCVZ': {  # 6965 cells
        'nemo': [
            ('GW20_34_PFCVZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_34_PFCVZ/GW20_34_PFCVZ.mex.tar.gz'),
        ],
    },
    'GW2034_V1': {  # 6756 cells
        'nemo': [
            ('GW20_34_V1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_34_V1/GW20_34_V1.mex.tar.gz'),
        ],
    },
    'GW2034_motor': {  # 51 cells
        'nemo': [
            ('GW20_34_motor', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_34_motor/GW20_34_motor.mex.tar.gz'),
        ],
    },
    'GW2034_parietal': {  # 9092 cells
        'nemo': [
            ('GW20_34_parietal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_34_parietal/GW20_34_parietal.mex.tar.gz'),
        ],
    },
    'GW2034_parietalVZ': {  # 3 cells
        'nemo': [
            ('GW20_34_ParVZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_34_ParVZ/GW20_34_ParVZ.mex.tar.gz'),
        ],
    },
    'GW20_PFC': {  # 4400 cells
        'nemo': [
            ('GW20PFC', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20PFC/GW20PFC.mex.tar.gz'),
        ],
    },
    'GW20_V1': {  # 12634 cells
        'nemo': [
            ('GW20V1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20V1/GW20V1.mex.tar.gz'),
        ],
    },
    'GW20_motor': {  # 11489 cells
        'nemo': [
            ('GW20_motor', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_motor/GW20_motor.mex.tar.gz'),
        ],
    },
    'GW20_somatosensory': {  # 8687 cells
        'nemo': [
            ('GW20_somato', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW20_somato/GW20_somato.mex.tar.gz'),
        ],
    },
    'GW22L_PFC': {  # 495 cells
        'nemo': [
            ('GW22_L_PFC1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22_L_PFC1/GW22_L_PFC1.mex.tar.gz'),
            ('GW22_L_PFC2', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22_L_PFC2/GW22_L_PFC2.mex.tar.gz'),
        ],
    },
    'GW22T_PFC': {  # 53 cells
        'nemo': [
            ('GW22T_PFC1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22T_PFC1/GW22T_PFC1.mex.tar.gz'),
            ('GW22T_PFC2', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22T_PFC2/GW22T_PFC2.mex.tar.gz'),
        ],
    },
    'GW22T_motor': {  # 3616 cells
        'nemo': [
            ('GW22T_motor1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22T_motor1/GW22T_motor1.mex.tar.gz'),
            ('GW22T_motor2', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22T_motor2/GW22T_motor2.mex.tar.gz'),
        ],
    },
    'GW22T_parietal': {  # 3759 cells
        'nemo': [
            ('GW22T_parietal1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22T_parietal1/GW22T_parietal1.mex.tar.gz'),
            ('GW22T_parietal2', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22T_parietal2/GW22T_parietal2.mex.tar.gz'),
        ],
    },
    'GW22T_somato': {  # 4600 cells
        'nemo': [
            ('GW22T_somato1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22T_somato1/GW22T_somato1.mex.tar.gz'),
            ('GW22T_somato2', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22T_somato2/GW22T_somato2.mex.tar.gz'),
        ],
    },
    'GW22_PFC': {  # 7109 cells
        'nemo': [
            ('GW22_PFC1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22_PFC1/GW22_PFC1.mex.tar.gz'),
            ('GW22_PFC2', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22_PFC2/GW22_PFC2.mex.tar.gz'),
        ],
    },
    'GW22_V1': {  # 4019 cells
        'nemo': [
            ('GW22_V1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22_V1/GW22_V1.mex.tar.gz'),
        ],
    },
    'GW22_motor': {  # 4330 cells
        'nemo': [
            ('GW22_motor1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22_motor1/GW22_motor1.mex.tar.gz'),
            ('GW22_motor2', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22_motor2/GW22_motor2.mex.tar.gz'),
        ],
    },
    'GW22_parietal': {  # 3008 cells
        'nemo': [
            ('GW22_parietal1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22_parietal1/GW22_parietal1.mex.tar.gz'),
            ('GW22_parietal2', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22_parietal2/GW22_parietal2.mex.tar.gz'),
        ],
    },
    'GW22_somato': {  # 7397 cells
        'nemo': [
            ('GW22_somato1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22_somato1/GW22_somato1.mex.tar.gz'),
            ('GW22_somato2', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW22_somato2/GW22_somato2.mex.tar.gz'),
        ],
    },
    'GW25_PFC': {  # 2068 cells
        'nemo': [
            ('GW25_PFC', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_PFC/GW25_PFC.mex.tar.gz'),
        ],
    },
    'GW25_PFCDL': {  # 8101 cells
        'nemo': [
            ('GW25_PFCDL', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_PFCDL/GW25_PFCDL.mex.tar.gz'),
        ],
    },
    'GW25_PFCMZ': {  # 9869 cells
        'nemo': [
            ('GW25_PFCMZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_PFCMZ/GW25_PFCMZ.mex.tar.gz'),
        ],
    },
    'GW25_PFCUL': {  # 516 cells
        'nemo': [
            ('GW25_PFCUL', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_PFCUL/GW25_PFCUL.mex.tar.gz'),
        ],
    },
    'GW25_PFCVZSVZ': {  # 1851 cells
        'nemo': [
            ('GW25_PFCVZ_SVZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_PFCVZ_SVZ/GW25_PFCVZ_SVZ.mex.tar.gz'),
        ],
    },
    'GW25_motor': {  # 2281 cells
        'nemo': [
            ('GW25_motor', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_motor/GW25_motor.mex.tar.gz'),
        ],
    },
    'GW25_parietal': {  # 10255 cells
        'nemo': [
            ('GW25_parietal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_parietal/GW25_parietal.mex.tar.gz'),
        ],
    },
    'GW25_parietalDL': {  # 6415 cells
        'nemo': [
            ('GW25_ParDL', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_ParDL/GW25_ParDL.mex.tar.gz'),
        ],
    },
    'GW25_parietalMZ': {  # 3404 cells
        'nemo': [
            ('GW25_ParMZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_ParMZ/GW25_ParMZ.mex.tar.gz'),
        ],
    },
    'GW25_parietalOSVZ': {  # 3766 cells
        'nemo': [
            ('GW25_ParOSVZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_ParOSVZ/GW25_ParOSVZ.mex.tar.gz'),
        ],
    },
    'GW25_parietalUL': {  # 2837 cells
        'nemo': [
            ('GW25_ParUL', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_ParUL/GW25_ParUL.mex.tar.gz'),
        ],
    },
    'GW25_parietalVZ': {  # 7638 cells
        'nemo': [
            ('GW25_ParVZ', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_ParVZ/GW25_ParVZ.mex.tar.gz'),
        ],
    },
    'GW25_somatosensory': {  # 9448 cells
        'nemo': [
            ('GW25_somato', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_somato/GW25_somato.mex.tar.gz'),
        ],
    },
    'GW25_temporal': {  # 1305 cells
        'nemo': [
            ('GW25_temporal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW25_temporal/GW25_temporal.mex.tar.gz'),
        ],
    },
    'gw17_PFC': {  # 1072 cells
        'nemo': [
            ('GW17_PFC', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW17_PFC/GRCh38/GRCh38.mex.tar.gz'),
        ],
    },
    'gw17_V1': {  # 1317 cells
        'nemo': [
            ('GW17_V1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW17_V1/GRCh38/GRCh38.mex.tar.gz'),
        ],
    },
    'gw17_motor': {  # 1377 cells
        'nemo': [
            ('GW17_motor', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW17_motor/GRCh38/GRCh38.mex.tar.gz'),
        ],
    },
    'gw17_parietal': {  # 18 cells
        'nemo': [
            ('GW17_parietal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW17_parietal/GRCh38/GRCh38.mex.tar.gz'),
        ],
    },
    'gw17_somato': {  # 228 cells
        'nemo': [
            ('GW17_somato', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW17_somato/GRCh38/GRCh38.mex.tar.gz'),
        ],
    },
    'gw19_2_Motor': {  # 11178 cells
        'nemo': [
            ('GW19_2_Motor', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_2_Motor/filtered_feature_bc_matrix/filtered_feature_bc_matrix.mex.tar.gz'),
        ],
    },
    'gw19_2_PFC': {  # 7912 cells
        'nemo': [
            ('GW19_2_PFC', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_2_PFC/filtered_feature_bc_matrix/filtered_feature_bc_matrix.mex.tar.gz'),
        ],
    },
    'gw19_2_Parietal': {  # 5140 cells
        'nemo': [
            ('GW19_2_Parietal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_2_Parietal/filtered_feature_bc_matrix/filtered_feature_bc_matrix.mex.tar.gz'),
        ],
    },
    'gw19_2_SS': {  # 7360 cells
        'nemo': [
            ('GW19_2_SS', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_2_SS/filtered_feature_bc_matrix/filtered_feature_bc_matrix.mex.tar.gz'),
        ],
    },
    'gw19_2_Temporal': {  # 8068 cells
        'nemo': [
            ('GW19_2_Temporal', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_2_Temporal/filtered_feature_bc_matrix/filtered_feature_bc_matrix.mex.tar.gz'),
        ],
    },
    'gw19_2_V1': {  # 4007 cells
        'nemo': [
            ('GW19_2_V1', 'https://data.nemoarchive.org/biccn/grant/u01_devhu/kriegstein/transcriptome/scell/10x_v2/human/processed/counts/GW19_2_V1/filtered_feature_bc_matrix/filtered_feature_bc_matrix.mex.tar.gz'),
        ],
    },
}

print(f'Samples: {len(SAMPLES)}')
print(f'NeMO folders: {sum(len(v["nemo"]) for v in SAMPLES.values())}')

Samples: 74
NeMO folders: 83


## 5. Download All Tarballs

Sequential downloads with skip-if-exists. Each failure is logged but doesn't halt the loop.
Expected: ~3.3 GB total, ~5–15 min on Colab.

In [5]:
import os, time, urllib.request, urllib.error

def download(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return 'skip', os.path.getsize(dest)
    tmp = dest + '.part'
    t0 = time.time()
    urllib.request.urlretrieve(url, tmp)
    os.rename(tmp, dest)
    return 'ok', os.path.getsize(dest)

failures = []
total_bytes = 0
for i, (ucsc, info) in enumerate(sorted(SAMPLES.items())):
    for nemo_folder, url in info['nemo']:
        dest = os.path.join(TARBALL_DIR, f'{nemo_folder}.mex.tar.gz')
        try:
            status, size = download(url, dest)
            total_bytes += size
            print(f'[{i+1:>2}/{len(SAMPLES)}] {status:4s} {nemo_folder:30s} {size/1e6:>7.1f} MB')
        except Exception as e:
            failures.append((nemo_folder, str(e)))
            print(f'[{i+1:>2}/{len(SAMPLES)}] FAIL {nemo_folder}: {e}')

print(f'\nTotal on disk: {total_bytes/1e9:.2f} GB, failures: {len(failures)}')
for f, err in failures: print(f'  {f}: {err}')

[ 1/74] skip GW14_occipital                    29.3 MB
[ 2/74] skip GW14_motor                        27.2 MB
[ 3/74] skip GW14_somato                       28.0 MB
[ 4/74] skip GW18_2_ParVZ                      77.3 MB
[ 5/74] skip GW18_2_TempVZ                     61.8 MB
[ 6/74] skip GW18_2_V1VZ                       77.0 MB
[ 7/74] skip GW18_2_motor                      52.2 MB
[ 8/74] skip GW18_2_motorVZ                    33.1 MB
[ 9/74] skip GW18_2_parietal                   61.7 MB
[10/74] skip GW18_2_somato                     53.0 MB
[11/74] skip GW18_2_somatoVZ                   26.0 MB
[12/74] skip GW18_2_temporal                   54.2 MB
[13/74] skip GW18_PFC                          57.3 MB
[14/74] skip GW18_V1                           51.9 MB
[15/74] skip GW18_motor                        59.2 MB
[16/74] skip GW18_parietal                     32.0 MB
[17/74] skip GW19_PFC_CP                       28.2 MB
[18/74] skip GW19_PFC_all                      18.4 MB
[19/74] sk

## 6. Extract All Tarballs

Each tarball extracts into its own subdirectory. MEX files (`matrix.mtx.gz`,
`barcodes.tsv.gz`, `features.tsv.gz` or `genes.tsv.gz`) are located by recursive search
since internal tarball layouts vary.

In [6]:
import tarfile, glob

def extract_tarball(tarball_path, extract_root):
    name = os.path.basename(tarball_path).replace('.mex.tar.gz','')
    dest = os.path.join(extract_root, name)
    if os.path.isdir(dest) and os.listdir(dest):
        return dest, 'skip'
    os.makedirs(dest, exist_ok=True)
    with tarfile.open(tarball_path, 'r:gz') as tf:
        tf.extractall(dest)
    return dest, 'ok'

def find_mex_dir(root):
    '''Find the directory containing matrix.mtx(.gz).'''
    for path in glob.glob(os.path.join(root, '**', 'matrix.mtx*'), recursive=True):
        return os.path.dirname(path)
    return None

extract_status = {}
for tarball in sorted(glob.glob(os.path.join(TARBALL_DIR, '*.mex.tar.gz'))):
    folder = os.path.basename(tarball).replace('.mex.tar.gz','')
    dest, status = extract_tarball(tarball, EXTRACT_DIR)
    mex_dir = find_mex_dir(dest)
    extract_status[folder] = mex_dir
    print(f'[{status}] {folder:30s} -> {mex_dir}')

missing = [f for f, m in extract_status.items() if m is None]
print(f'\nExtracted: {len(extract_status) - len(missing)}/{len(extract_status)}')
if missing:
    print('Missing MEX files:', missing)

[skip] GW14_motor                     -> /content/bhaduri2021/extracted/GW14_motor/GRCh38
[skip] GW14_occipital                 -> /content/bhaduri2021/extracted/GW14_occipital/GRCh38
[skip] GW14_somato                    -> /content/bhaduri2021/extracted/GW14_somato/GRCh38
[skip] GW17_PFC                       -> /content/bhaduri2021/extracted/GW17_PFC/GRCh38
[skip] GW17_V1                        -> /content/bhaduri2021/extracted/GW17_V1/GRCh38
[skip] GW17_motor                     -> /content/bhaduri2021/extracted/GW17_motor/GRCh38
[skip] GW17_parietal                  -> /content/bhaduri2021/extracted/GW17_parietal/GRCh38
[skip] GW17_somato                    -> /content/bhaduri2021/extracted/GW17_somato/GRCh38
[skip] GW18_2_ParVZ                   -> /content/bhaduri2021/extracted/GW18_2_ParVZ/GW18_2_ParVZ
[skip] GW18_2_TempVZ                  -> /content/bhaduri2021/extracted/GW18_2_TempVZ/GW18_2_TempVZ
[skip] GW18_2_V1VZ                    -> /content/bhaduri2021/extracted/GW18_2

## 7. Load Per-Sample AnnDatas

For each UCSC sample:
1. Load each NeMO folder's MEX via `sc.read_10x_mtx`
2. Strip 10x `-N` suffix from barcodes
3. Prepend the UCSC prefix to match `Cell Name` format
4. For split-lane samples (GW22 series), concat the two tarballs
5. Attach per-sample obs columns (`donor`, `area`, `age_gw`, `source_tarball`)


  ### 7.0 Robust MEX Loader (helper) - subsequently added upon failing performance of this section                                                                                                                                                                    
                                                                                                                                                                                                         
  Defines `read_mex()`, used in place of `sc.read_10x_mtx()` in the loading loop below. The default scanpy reader assumes a clean 10x folder (`matrix.mtx.gz` + `barcodes.tsv.gz` + `features.tsv.gz`).  
  The Bhaduri 2021 tarballs don't follow that convention — see the diagnostics at the end of the chapter for the actual file layout. This helper accepts:                                                
                                                                                                                                                                                                         
  - `.mtx` and `.tsv` either gzipped or uncompressed,                                                                                                                                                    
  - `features.tsv*` **or** `genes.tsv*` for the gene file,
  - a glob fallback for the gene file when the filename is malformed (catches the `enes.tsv` typo found in some folders).                                                                                
                                                                                                                                                                                                         
  Returns a `cells × genes` AnnData with `var_names_make_unique()` already applied. Imports `scipy.sparse as sp` because the helper builds the CSR matrix itself rather than going through a scanpy      
  wrapper.   

In [7]:
import os, glob
import pandas as pd
import scipy.io as sio
import anndata as ad

def _find(mex_dir, base_pattern):
    for ext in ('', '.gz'):
        hits = glob.glob(os.path.join(mex_dir, f'{base_pattern}{ext}'))
        if hits:
            return hits[0]
    return None

def read_mex(mex_dir):
    mtx_path  = _find(mex_dir, 'matrix.mtx')
    bc_path   = _find(mex_dir, 'barcodes.tsv')
    feat_path = _find(mex_dir, 'features.tsv') or _find(mex_dir, 'genes.tsv')
    if feat_path is None:
        cand = [p for p in glob.glob(os.path.join(mex_dir, '*.tsv*'))
                if 'barcodes' not in os.path.basename(p)]
        feat_path = cand[0] if cand else None
    if not (mtx_path and bc_path and feat_path):
        raise FileNotFoundError(f'Missing MEX files in {mex_dir}: mtx={mtx_path}, bc={bc_path}, feat={feat_path}')
    X = sp.csr_matrix(sio.mmread(mtx_path).T)
    barcodes = pd.read_csv(bc_path, header=None, sep='\t')[0].astype(str).tolist()
    feat_df  = pd.read_csv(feat_path, header=None, sep='\t')
    gene_names = feat_df[1].astype(str).tolist() if feat_df.shape[1] >= 2 else feat_df[0].astype(str).tolist()
    a = ad.AnnData(X=X, obs=pd.DataFrame(index=barcodes), var=pd.DataFrame(index=gene_names))
    a.var_names_make_unique()
    return a

In [9]:
import re
import anndata as ad
import scanpy as sc
import scipy.sparse as sp

def parse_donor_area(ucsc_prefix):
    '''Parse UCSC prefix into donor + area. Matches the irregular casing in the meta.tsv.'''
    p = ucsc_prefix
    # lowercase donors: gw17, gw19_2
    m = re.match(r'^(gw\d+(?:_\d+)?)_(.+)$', p)
    if m:
        return m.group(1).upper(), m.group(2)
    # GW2031 / GW2034 (no underscore in donor)
    m = re.match(r'^(GW20)(3[14])_(.+)$', p)
    if m:
        return f'{m.group(1)}_{m.group(2)}', m.group(3)
    # Standard: GW##[_2|T|L]_area
    m = re.match(r'^(GW\d+(?:_\d+|T|L)?)_(.+)$', p)
    if m:
        return m.group(1), m.group(2)
    return p, ''

def donor_to_gw(donor):
    m = re.match(r'^GW(\d+)', donor, re.IGNORECASE)
    return int(m.group(1)) if m else None

def load_sample(ucsc_prefix, nemo_entries):
    parts = []
    for nemo_folder, _ in nemo_entries:
        mex_dir = extract_status.get(nemo_folder)
        if mex_dir is None:
            print(f'  WARN: no MEX for {nemo_folder}')
            continue
        a = read_mex(mex_dir)
        # Strip 10x -N suffix, then prepend UCSC prefix to match Cell Name format
        a.obs_names = [f'{ucsc_prefix}_{bc.split(chr(45))[0]}' for bc in a.obs_names]
        a.obs['source_tarball'] = nemo_folder
        parts.append(a)
    if not parts:
        return None
    a = ad.concat(parts, axis=0, join='outer', merge='first', index_unique=None) if len(parts) > 1 else parts[0]
    donor, area = parse_donor_area(ucsc_prefix)
    a.obs['ucsc_prefix'] = ucsc_prefix
    a.obs['donor'] = donor
    a.obs['area']  = area
    a.obs['age_gw'] = donor_to_gw(donor)
    a.var_names_make_unique()
    a.obs_names_make_unique()
    return a

per_sample = {}
for i, (ucsc, info) in enumerate(sorted(SAMPLES.items())):
    a = load_sample(ucsc, info['nemo'])
    if a is None:
        print(f'[{i+1:>2}/{len(SAMPLES)}] SKIP {ucsc}')
        continue
    per_sample[ucsc] = a
    print(f'[{i+1:>2}/{len(SAMPLES)}] {ucsc:30s} {a.n_obs:>7d} cells, {a.n_vars:>6d} genes')

print(f'\nLoaded {len(per_sample)} samples')
print(f'Total raw cells: {sum(a.n_obs for a in per_sample.values()):,}')

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


[ 1/74] GW14_V1                          14200 cells,  33694 genes


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


[ 2/74] GW14_motor                       20053 cells,  33694 genes


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


...

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


[72/74] gw19_2_SS                        14728 cells,  33694 genes


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


[73/74] gw19_2_Temporal                  28162 cells,  33694 genes
[74/74] gw19_2_V1                         5694 cells,  33694 genes

Loaded 74 samples
Total raw cells: 1,003,812


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  ### 7.1 Diagnostic — Survey MEX Folder Contents                                                                                                                                                        
                                                                                                                                                                                                         
  After `sc.read_10x_mtx()` failed on `GW14_motor`, list the file contents of several extracted folders to characterize the problem. The output reveals two issues in the original Bhaduri lab archives:
                                                                                                                                                                                                         
  1. **Inconsistent naming** — some folders use `genes.tsv`, others `features.tsv`. Both are valid 10x conventions (Cell Ranger v2 vs v3) but `sc.read_10x_mtx()` expects one specific layout.           
  2. **Corrupted filename** — `GW14_motor` and `GW18_PFC` ship `enes.tsv` instead of `genes.tsv` (missing leading `g`). The malformed name is the actual cause of the loader crash.

In [11]:
import os
for d in ['GW14_V1', 'GW14_motor', 'GW14_occipital', 'GW19_2_Motor', 'GW18_PFC']:
    p = f'/content/bhaduri2021/extracted/{d}'
    for root, _, files in os.walk(p):
        if any('matrix' in f for f in files):
            print(f'{d}:')
            for f in sorted(files): print(f'   {f}')
            break

GW14_motor:
   barcodes.tsv
   enes.tsv
   matrix.mtx
GW14_occipital:
   barcodes.tsv
   genes.tsv
   matrix.mtx
GW19_2_Motor:
   barcodes.tsv
   features.tsv
   matrix.mtx
GW18_PFC:
   barcodes.tsv
   enes.tsv
   matrix.mtx


  ### 7.2 Diagnostic — File Listing for the Anomalous Folder                                                                                                                                             
                                                                                                                                                                                                         
  `ls -la` on `GW14_motor` to confirm the `enes.tsv` filename isn't a display artifact and to check file metadata. Output shows:                                                                         
                                                                                                                                                                                                         
  - `enes.tsv` is dated **Apr 12 2019** — the typo is in the original archive uploaded to NeMO, not introduced by our extraction.                                                                        
  - File size (840 KB) and matrix size (107 MB) are reasonable for ~7k cells × ~33k genes — the data itself is intact, only the filename is wrong.
                                                                                                                                                                                                         
  This confirms the fix belongs in our loader (glob fallback for the gene file) rather than in a renaming/repair step on disk.      

In [12]:
!ls -la /content/bhaduri2021/extracted/GW14_motor/GRCh38/

total 106080
drwxr-xr-x 2 5030 10016      4096 Oct  9  2020 .
drwxr-xr-x 3 root root       4096 Apr 18 19:03 ..
-rw-r--r-- 1 5030 10016    381007 Apr 12  2019 barcodes.tsv
-rw-r--r-- 1 5030 10016    840938 Apr 12  2019 enes.tsv
-rw-r--r-- 1 5030 10016 107383045 Apr 12  2019 matrix.mtx


## 8. Concatenate All Samples

Inner gene join (keep only genes present in every sample). Var names should be consistent
since all samples came from the same GRCh38 reference, but guard with inner join anyway.

In [13]:
adata = ad.concat(list(per_sample.values()), axis=0, join='inner', merge='first', index_unique=None)
adata.obs_names_make_unique()
print(f'Concatenated: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
print(f'Donors:  {sorted(adata.obs["donor"].unique())}')
print(f'Areas:   {sorted(adata.obs["area"].unique())}')
print(f'GW:      {sorted(adata.obs["age_gw"].unique())}')
# Free per-sample memory
del per_sample
import gc; gc.collect()

Concatenated: 1,003,812 cells x 33,694 genes
Donors:  ['GW14', 'GW17', 'GW18', 'GW18_2', 'GW19', 'GW19_2', 'GW20', 'GW20_31', 'GW20_34', 'GW22', 'GW22L', 'GW22T', 'GW25']
Areas:   ['Motor', 'PFC', 'PFCDL', 'PFCMZ', 'PFCUL', 'PFCVZ', 'PFCVZSVZ', 'PFC_CP', 'PFC_all', 'Parietal', 'ParietalVZ', 'SS', 'Temporal', 'TemporalVZ', 'V1', 'V1VZ', 'V1_CP', 'V1_all', 'motor', 'motorVZ', 'motor_CP', 'motor_all', 'parietal', 'parietalDL', 'parietalMZ', 'parietalOSVZ', 'parietalUL', 'parietalVZ', 'somato', 'somatoVZ', 'somatosensory', 'temporal']
GW:      [np.int64(14), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(22), np.int64(25)]


937

## 9. Download and Merge UCSC Annotations

UCSC `meta.tsv` carries the authors' post-QC cell-type annotations. We match on `Cell Name`
(which our `obs_names` now mirror) and inner-join — cells without annotations are dropped.

In [14]:
import pandas as pd, urllib.request

if not os.path.exists(UCSC_META_LOCAL):
    print(f'Downloading UCSC meta.tsv ({UCSC_META_URL}) ...')
    urllib.request.urlretrieve(UCSC_META_URL, UCSC_META_LOCAL)
print(f'Size: {os.path.getsize(UCSC_META_LOCAL)/1e6:.1f} MB')

meta = pd.read_csv(UCSC_META_LOCAL, sep='\t', index_col=0)
meta.index.name = None
print(f'UCSC rows: {len(meta):,}')
print(f'Columns: {list(meta.columns)}')

Size: 46.5 MB
UCSC rows: 404,218
Columns: ['CellType', 'Name', 'CombinedCluster - Iteration 1', 'CombinedCluster - Final', 'Area', 'Age', 'Lamina', 'Individual', 'ConsensusCellType - Final', 'AgeRange', 'Main Brain Region']


In [15]:
# Sanity: overlap between our barcodes and UCSC Cell Names
our_set = set(adata.obs_names)
ucsc_set = set(meta.index)
overlap = our_set & ucsc_set
print(f'Our cells:      {len(our_set):,}')
print(f'UCSC cells:     {len(ucsc_set):,}')
print(f'Overlap:        {len(overlap):,}  ({100*len(overlap)/len(ucsc_set):.1f}% of UCSC)')
print(f'Our unmatched:  {len(our_set - ucsc_set):,}')
print(f'UCSC unmatched: {len(ucsc_set - our_set):,}')

if len(overlap) == 0:
    raise RuntimeError('Zero overlap — barcode format mismatch. Inspect obs_names vs meta.index.')

Our cells:      1,003,812
UCSC cells:     404,218
Overlap:        396,186  (98.0% of UCSC)
Our unmatched:  607,626
UCSC unmatched: 8,032


In [16]:
# Inner-join: keep only annotated cells
keep = adata.obs_names.isin(meta.index)
adata = adata[keep].copy()
adata.obs = adata.obs.join(meta, how='left')

# Rename for downstream convenience
adata.obs = adata.obs.rename(columns={
    'CellType': 'cell_type_coarse',
    'ConsensusCellType - Final': 'cell_type',
    'CombinedCluster - Final': 'cluster_final',
    'Area': 'area_ucsc',
    'Age': 'age_gw_ucsc',
    'Individual': 'individual',
    'Lamina': 'lamina',
    'AgeRange': 'age_range',
})

print(f'After annotation merge: {adata.n_obs:,} cells x {adata.n_vars:,} genes')

After annotation merge: 396,186 cells x 33,694 genes


## 10. Sanity Checks

In [17]:
print('Cell counts by donor x area:')
print(pd.crosstab(adata.obs['donor'], adata.obs['area']))
print('\nCoarse cell type distribution:')
print(adata.obs['cell_type_coarse'].value_counts())
print('\nConsensus cell type (top 20):')
print(adata.obs['cell_type'].value_counts().head(20))
print(f'\nn_genes min/median/max: {adata.X.getnnz(axis=1).min()} / '
      f'{int(pd.Series(adata.X.getnnz(axis=1)).median())} / '
      f'{adata.X.getnnz(axis=1).max()}')

Cell counts by donor x area:
area     Motor    PFC  PFCDL  PFCMZ  PFCUL  PFCVZ  PFCVZSVZ  PFC_CP  PFC_all  \
donor                                                                          
GW14         0      0      0      0      0      0         0       0        0   
GW17         0   1072      0      0      0      0         0       0        0   
GW18         0  13785      0      0      0      0         0       0        0   
GW18_2       0      0      0      0      0      0         0       0        0   
GW19         0      0      0      0      0      0         0    3731     3565   
GW19_2   11178   7912      0      0      0      0         0       0        0   
GW20         0   4400      0      0      0      0         0       0        0   
GW20_31      0   6681      0      0      0      0         0       0        0   
GW20_34      0   3243      0      0      0   6965         0       0        0   
GW22         0   7109      0      0      0      0         0       0        0   
GW22L      

## 11. Save to Drive

In [18]:
adata.write_h5ad(OUT_H5AD, compression='gzip')
size_gb = os.path.getsize(OUT_H5AD) / 1e9
print(f'Saved: {OUT_H5AD} ({size_gb:.2f} GB)')

Saved: /content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2021/bhaduri_2021_raw.h5ad (1.40 GB)


## Done

Next: `colab_07_preprocessing.ipynb` — QC filtering, normalization, HVG on the 2 datasets
(Bhaduri 2020 organoids + Bhaduri 2021 fetal).
